In [1]:
# Supreme

### Bank Customer Churn Prediction Project 

##### Prepared By : Rahangdale Vikas Dindayal

In [ ]:
# Dataset 

https://www.kaggle.com/datasets/santoshd3/bank-customers/data

#### Solution : 

In [2]:
# Importing required libraries 

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint




In [ ]:
# Loading of dataset 

df=pd.read_csv("Churn_Modeling.csv")
df

### EDA 

In [ ]:
df.sample(4)

In [ ]:
# dropping irrelevant columns
df.drop(["RowNumber","CustomerId","Surname"],axis=1, inplace=True)
df

In [ ]:
df.info()

In [ ]:
# Missing Value checking 

df.isnull().sum()

In [ ]:
# checking of duplicate records 

df.duplicated().sum()

In [ ]:
# Stats

df.describe().T


In [ ]:
# Checking data types 

df.dtypes

In [ ]:
# Checking target class  distribution 

df["Exited"].value_counts()

In [ ]:
# Making a copy so the original dataframe remains available if needed
df = df.copy()

# Encoding of  Gender (binary categorical → Label Encoding)
le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])  # Male=1, Female=0 (or vice versa)

# Encoding of  Geography (multi-class categorical → One-Hot Encoding)

df = pd.get_dummies(df, columns=["Geography"], drop_first=True)
# drop_first=True avoids dummy variable trap

df.head()

In [ ]:
# Distribution visualisation for all variables in df
cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c])]

n_cols = 3
n_rows = (len(cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    ax = axes[i]
    
    if df[col].nunique() <= 10:
        sns.countplot(data=df, x=col, ax=ax, color="steelblue")
    else:
        sns.histplot(data=df, x=col, kde=True, bins="auto", ax=ax, color="steelblue")
    
    ax.set_title(f"{col} distribution")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")

for j in range(len(cols), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Outlier visualization for each variable
cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c])]

n_cols = 3
n_rows = (len(cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    ax = axes[i]
    sns.boxplot(data=df, x=col, ax=ax, color="steelblue", width=0.5, fliersize=3)
    ax.set_title(f"{col} outlier boxplot")
    ax.set_xlabel(col)
    ax.set_ylabel("Value")

for j in range(len(cols), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap visualization
plt.figure(figsize=(12, 10))
sns.heatmap(
    df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    vmin=-1,
    vmax=1
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Skewness checking visualization

feature_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c])]
feature_df = df[feature_cols].astype(float)

skew_vals = feature_df.skew().sort_values(ascending=False)

# Bar chart of skewness values
plt.figure(figsize=(10, 4))
sns.barplot(x=skew_vals.index, y=skew_vals.values, palette="coolwarm")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Feature Skewness")
plt.ylabel("Skewness")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Individual distribution plots for each feature
for col in feature_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(feature_df[col], kde=True, bins="auto", color="steelblue")
    plt.axvline(feature_df[col].mean(), color="red", linestyle="--", linewidth=1, label="Mean")
    plt.title(f"{col} | skewness = {skew_vals[col]:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.preprocessing import PowerTransformer

# Working on a copy so the original dataframe is preserved
df = df.copy()

# Separation of  feature matrix from target, keeping target untouched
X = df.drop(columns=["Exited"]).copy()
y = df["Exited"].copy()



In [ ]:
# Selecting  only continuous numeric features for skewness treatment
numeric_cols = X.select_dtypes(include=["number"]).columns
skew_vals = X[numeric_cols].skew().abs()

# Only transforming  features with meaningful skewness and enough unique values
skewed_cols = [
    col for col in skew_vals.index
    if skew_vals[col] > 0.5 and X[col].nunique() > 10
]

if skewed_cols:
    pt = PowerTransformer(method="yeo-johnson", standardize=False)
    X[skewed_cols] = pt.fit_transform(X[skewed_cols])
    print("Applied skewness correction to:", skewed_cols)
else:
    print("No continuous skewed features found above the threshold.")



In [ ]:
# Rebuilding  df with transformed features and the original target
df = pd.concat([X, y], axis=1)


In [ ]:
# Verifying the target variable was not changed
df["Exited"].value_counts()

In [ ]:
# Visualization of  skewness after treatment
numeric_cols = X.select_dtypes(include=["number"]).columns
skew_after = X[numeric_cols].skew().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(x=skew_after.index, y=skew_after.values, palette="coolwarm")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Feature Skewness After Power Transformation")
plt.ylabel("Skewness")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# individual distribution plots for each feature after treatment
for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(X[col], kde=True, bins="auto", color="steelblue")
    plt.axvline(X[col].mean(), color="red", linestyle="--", linewidth=1, label="Mean")
    plt.title(f"{col} | skewness = {skew_after[col]:.2f}")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Comparison of outlier counts before vs after skewness treatment
orig_numeric = feature_df[numeric_cols].copy()
treated_numeric = X[numeric_cols].copy()

def count_iqr_outliers(frame):
    counts = {}
    for col in frame.columns:
        q1 = frame[col].quantile(0.25)
        q3 = frame[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        counts[col] = int(((frame[col] < lower) | (frame[col] > upper)).sum())
    return pd.Series(counts)

before_treatment = count_iqr_outliers(orig_numeric)
after_treatment = count_iqr_outliers(treated_numeric)

compare = pd.DataFrame({
    "Before_treatment": before_treatment,
    "After_treatment": after_treatment
}).sort_values(["Before_treatment", "After_treatment"], ascending=False)



In [ ]:
# visualisation of outlier before anf after skewness treatment
plt.figure(figsize=(12, 5))
compare.plot(kind="bar")
plt.title("Outlier count comparison before vs after skewness treatment")
plt.ylabel("Outlier count")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
# Boxplot visualization after skewness treatment
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=treated_numeric, x=col, ax=axes[i], color="steelblue", width=0.5, fliersize=3)
    axes[i].set_title(f"{col} boxplot after treatment")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Value")

for j in range(len(numeric_cols), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# IQR-based outlier treatment (winsorization) for continuous features only
X = X.copy()
y = y.copy()

outlier_cols = [
    col for col in X.columns
    if pd.api.types.is_numeric_dtype(X[col])
    and col not in {"Gender", "HasCrCard", "IsActiveMember", "Geography_Germany", "Geography_Spain"}
]

def count_iqr_outliers(frame, cols):
    counts = {}
    for col in cols:
        q1 = frame[col].quantile(0.25)
        q3 = frame[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        counts[col] = int(((frame[col] < lower) | (frame[col] > upper)).sum())
    return pd.Series(counts)

outlier_before = count_iqr_outliers(X, outlier_cols)

for col in outlier_cols:
    q1 = X[col].quantile(0.25)
    q3 = X[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    X[col] = X[col].clip(lower=lower, upper=upper)

outlier_after = count_iqr_outliers(X, outlier_cols)

outlier_summary = pd.DataFrame({
    "Before_treatment": outlier_before,
    "After_treatment": outlier_after
}).sort_values(["Before_treatment", "After_treatment"], ascending=False)

print(outlier_summary)


In [ ]:
df = pd.concat([X, y], axis=1)
df.head()

In [ ]:
# Visualization of outliers after treatment
n_cols = 3
n_rows = (len(outlier_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    sns.boxplot(data=X, x=col, ax=axes[i], color="steelblue", width=0.5, fliersize=3)
    axes[i].set_title(f"{col} boxplot after treatment")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Value")

for j in range(len(outlier_cols), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()



In [ ]:
# Showing  remaining outlier counts after clipping
remaining_outliers = count_iqr_outliers(X, outlier_cols)
print("Remaining outliers after treatment:")
print(remaining_outliers)

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



In [ ]:
# Applying  scaler only on training data and transform test data
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)



### Building  ANN with multiple hidden layers

In [ ]:
# Building  simple ANN structure 
ann_model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])


In [ ]:
ann_model.summary()


In [ ]:
ann_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_scaled.to_numpy(),
    y_train.to_numpy().astype("float32"),
    epochs=20,
    batch_size=32,
    validation_data=(X_test_scaled.to_numpy(), y_test.to_numpy().astype("float32")),
    class_weight=class_weight_dict if "class_weight_dict" in globals() else None,
    verbose=1
)


In [ ]:
# Evaluation 
test_loss, test_accuracy = ann_model.evaluate(
    X_test_scaled.to_numpy(),
    y_test.to_numpy().astype("float32"),
    verbose=0
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

###  Model with droupout and early stopping method 


In [ ]:
keras.backend.clear_session()

regularized_ann = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

In [ ]:
regularized_ann.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Early stopping 
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)


In [ ]:
# Model checkpoint 
checkpoint = ModelCheckpoint(
    "best_dropout_ann.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)


In [ ]:
# Prediction
history_dropout = regularized_ann.fit(
    X_train_scaled.to_numpy(),
    y_train.to_numpy().astype("float32"),
    epochs=60,
    batch_size=32,
    validation_data=(X_test_scaled.to_numpy(), y_test.to_numpy().astype("float32")),
    callbacks=[early_stopping, checkpoint],
    class_weight=class_weight_dict if "class_weight_dict" in globals() else None,
    verbose=1
)


In [ ]:
# Evaluation
best_dropout_ann = keras.models.load_model("best_dropout_ann.keras")

test_loss, test_accuracy = best_dropout_ann.evaluate(
    X_test_scaled.to_numpy(),
    y_test.to_numpy().astype("float32"),
    verbose=0
)

print(f"Regularized ANN Test Loss: {test_loss:.4f}")
print(f"Regularized ANN Test Accuracy: {test_accuracy:.4f}")

###  Application of Keras tunner 

In [ ]:
import keras_tuner as kt


def build_tuned_model(hp):
    model = keras.Sequential()

    n_hidden_layers = hp.Int("n_hidden_layers", min_value=1, max_value=4)

    for i in range(n_hidden_layers):
        units = hp.Int(f"units_{i}", min_value=32, max_value=256, step=32)
        activation = hp.Choice(f"activation_{i}", values=["relu", "elu", "tanh"])
        dropout = hp.Float(f"dropout_{i}", min_value=0.0, max_value=0.4, step=0.1)

        if i == 0:
            model.add(Dense(units, activation=activation, input_shape=(X_train_scaled.shape[1],)))
        else:
            model.add(Dense(units, activation=activation))

        model.add(BatchNormalization())
        model.add(Dropout(dropout))

    model.add(Dense(1, activation="sigmoid"))

    optimizer_name = hp.Choice("optimizer", values=["adam", "rmsprop", "nadam"])
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")

    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "rmsprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.Nadam(learning_rate=learning_rate)

    loss_name = hp.Choice("loss", values=["binary_crossentropy", "binary_focal_crossentropy"])
    if loss_name == "binary_crossentropy":
        loss_fn = keras.losses.BinaryCrossentropy(from_logits=False)
    else:
        loss_fn = keras.losses.BinaryFocalCrossentropy(from_logits=False, alpha=0.25, gamma=2.0)

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])
    return model



In [ ]:
keras.backend.clear_session()

In [ ]:
tuner = kt.Hyperband(
    build_tuned_model,
    objective="val_accuracy",
    max_epochs=25,
    factor=3,
    directory="kt_tuning",
    project_name="churn_ann_tuning",
    overwrite=True
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=0
)

In [ ]:
from sklearn.utils import class_weight
class_weights_array = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dictionary format
class_weight_dict = dict(enumerate(class_weights_array))

print("Class Weights:", class_weight_dict)

In [ ]:
tuner.search(
    X_train_scaled.to_numpy(),
    y_train.to_numpy().astype("float32"),
    validation_data=(X_test_scaled.to_numpy(), y_test.to_numpy().astype("float32")),
    epochs=25,
    callbacks=[early_stopping],
    class_weight=class_weight_dict,
    verbose=0
)

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
best_tuned_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
print("Best hyperparameters:")
for key, value in best_hp.values.items():
    print(f"{key}: {value}")

best_tuned_model.summary()

test_loss, test_accuracy = best_tuned_model.evaluate(
    X_test_scaled.to_numpy(),
    y_test.to_numpy().astype("float32"),
    verbose=0
)

print(f"Best tuned model test loss: {test_loss:.4f}")
print(f"Best tuned model test accuracy: {test_accuracy:.4f}")

In [ ]:
# Comparison of  trained ANN models using test accuracy and keep the best one
model_map = {}

for name in ["ann_model", "best_dropout_ann", "best_tuned_model"]:
    model = globals().get(name)
    if model is not None:
        model_map[name] = model

if not model_map:
    raise ValueError("No trained ANN models are available in the notebook variables.")

results = []
for model_name, model in model_map.items():
    loss, accuracy = model.evaluate(
        X_test_scaled.to_numpy(),
        y_test.to_numpy().astype("float32"),
        verbose=0
    )
    results.append({
        "Model": model_name,
        "Test_Loss": float(loss),
        "Test_Accuracy": float(accuracy)
    })

comparison_df = pd.DataFrame(results).sort_values(
    by=["Test_Accuracy", "Test_Loss"],
    ascending=[False, True]
).reset_index(drop=True)

comparison_df

best_model_name = comparison_df.iloc[0]["Model"]
final_best_model = model_map[best_model_name]

print("Best model selected:", best_model_name)
print(f"Best test accuracy: {comparison_df.iloc[0]['Test_Accuracy']:.4f}")

## Comparison table for the three ANN variants

In [ ]:

model_map = {
    "Simple ANN": ann_model,
    "ANN with Dropout": best_dropout_ann,
    "Tuned ANN": best_tuned_model
}

results = []

for model_name, model in model_map.items():
    loss, accuracy = model.evaluate(
        X_test_scaled.to_numpy(),
        y_test.to_numpy().astype("float32"),
        verbose=0
    )
    results.append({
        "Model": model_name,
        "Test_Loss": float(loss),
        "Test_Accuracy": float(accuracy)
    })

comparison_df = pd.DataFrame(results).sort_values(
    by=["Test_Accuracy", "Test_Loss"],
    ascending=[False, True]
).reset_index(drop=True)

comparison_df